In [2]:
import pandas as pd
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv("tmdb_ds.csv")

print(df.columns.tolist())

['movie_id', 'title', 'cast', 'crew']


In [3]:
df["cast"] = df["cast"].fillna("")
df["title"] = df["title"].fillna("")

print(df[["title", "cast"]].head())

                                      title  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   
2                                   Spectre   
3                     The Dark Knight Rises   
4                               John Carter   

                                                cast  
0  [{"cast_id": 242, "character": "Jake Sully", "...  
1  [{"cast_id": 4, "character": "Captain Jack Spa...  
2  [{"cast_id": 1, "character": "James Bond", "cr...  
3  [{"cast_id": 2, "character": "Bruce Wayne / Ba...  
4  [{"cast_id": 5, "character": "John Carter", "c...  


In [4]:
stop_words = set(ENGLISH_STOP_WORDS)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

df["clean_text"] = df["title"].apply(clean_text)

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(df["clean_text"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (4803, 5000)


In [5]:
def recommend(item_name, top_n=5):
    matches = df.index[
        df["title"].str.strip().str.lower() == item_name.strip().lower()
    ].tolist()

    if not matches:
        print("Movie not found. Please enter a title from the dataset.")
        return pd.DataFrame(columns=["title", "similarity_score"])

    movie_index = matches[0]
    similarity_scores = cosine_similarity(
        tfidf_matrix[movie_index], tfidf_matrix
    ).flatten()

    similar_indices = similarity_scores.argsort()[::-1]

    similar_indices = [
        index for index in similar_indices if index != movie_index
    ][:top_n]

    recommendations = pd.DataFrame({
        "title": df.iloc[similar_indices]["title"].values,
        "similarity_score": similarity_scores[similar_indices]
    })

    return recommendations.reset_index(drop=True)

In [6]:
test_movies = df["title"].drop_duplicates().dropna().head(3).tolist()

for movie in test_movies:
    print("\nRecommendations for:", movie)
    display(recommend(movie, top_n=5))


Recommendations for: Avatar


,title,similarity_score
0,My Date with Drew,0.0
1,Shanghai Calling,0.0
2,"Signed, Sealed, Delivered",0.0
3,Newlyweds,0.0
4,El Mariachi,0.0



Recommendations for: Pirates of the Caribbean: At World's End


,title,similarity_score
0,The World's End,0.611208
1,Pirates of the Caribbean: On Stranger Tides,0.448735
2,Pirates of the Caribbean: Dead Man's Chest,0.384332
3,Pirates of the Caribbean: The Curse of the Bla...,0.370792
4,This Is the End,0.331857



Recommendations for: Spectre


,title,similarity_score
0,My Date with Drew,0.0
1,Shanghai Calling,0.0
2,"Signed, Sealed, Delivered",0.0
3,Newlyweds,0.0
4,El Mariachi,0.0


In [7]:
recommend("Avatar",top_n=5)

,title,similarity_score
0,My Date with Drew,0.0
1,Shanghai Calling,0.0
2,"Signed, Sealed, Delivered",0.0
3,Newlyweds,0.0
4,El Mariachi,0.0


In [9]:
recommend("Spectre",top_n=5)

,title,similarity_score
0,My Date with Drew,0.0
1,Shanghai Calling,0.0
2,"Signed, Sealed, Delivered",0.0
3,Newlyweds,0.0
4,El Mariachi,0.0


In [10]:
recommend("Titanic",top_n=5)

,title,similarity_score
0,Raise the Titanic,0.694649
1,The Chambermaid on the Titanic,0.545759
2,The Chronicles of Narnia: Prince Caspian,0.000000
3,Shanghai Calling,0.000000
4,El Mariachi,0.000000
